In [1]:
# =============================================================================
# CELL 1 — Imports
# =============================================================================

from __future__ import annotations

import importlib
import json
from pathlib import Path
from types import SimpleNamespace

import pandas as pd
from dotenv import find_dotenv, load_dotenv

import src.pago_pipeline.ncbi_metadata_snapshot as ncbi_metadata_snapshot_module

# Reload pipeline modules so notebook reruns pick up local code changes.
ncbi_metadata_snapshot_module = importlib.reload(ncbi_metadata_snapshot_module)

load_latest_metadata_snapshot = (
    ncbi_metadata_snapshot_module.load_latest_metadata_snapshot
)

from src.pago_pipeline.storage import read_json_file, sha256_of_file

In [2]:
# =============================================================================
# CELL 2 — Load environment and resolve project root
# =============================================================================

dotenv_path = find_dotenv(usecwd=False)

if not dotenv_path:
    raise FileNotFoundError(
        "Could not find a .env file while walking up parent directories. "
        "Place .env with your NCBI configuration at the project root."
    )

load_dotenv(dotenv_path=dotenv_path, override=True)

PROJECT_ROOT = Path(dotenv_path).resolve().parent

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\pago-proj\pAgo-project


In [3]:
# =============================================================================
# CELL 3 — Define QC inspection configuration
# =============================================================================

XML_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "01-raw" / "protein_xml_snapshots"
)
METADATA_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "02-intermediate" / "protein_metadata_csv"
)

print(f"XML snapshot root directory: {XML_SNAPSHOT_ROOT_DIRECTORY}")
print(f"Metadata snapshot root directory: {METADATA_SNAPSHOT_ROOT_DIRECTORY}")

XML snapshot root directory: C:\pago-proj\pAgo-project\data\01-raw\protein_xml_snapshots
Metadata snapshot root directory: C:\pago-proj\pAgo-project\data\02-intermediate\protein_metadata_csv


In [4]:
# =============================================================================
# CELL 4 — Resolve frozen metadata snapshot artifacts
# =============================================================================

metadata_snapshot_payload = load_latest_metadata_snapshot(
    snapshot_root_directory=METADATA_SNAPSHOT_ROOT_DIRECTORY,
)
metadata_snapshot_directory = metadata_snapshot_payload["snapshot_directory"]
METADATA_CSV_FILE_PATH = metadata_snapshot_payload["csv_file_path"]
METADATA_MANIFEST_FILE_PATH = metadata_snapshot_payload["manifest_file_path"]
QC_REPORT_FILE_PATH = metadata_snapshot_payload["qc_report_file_path"]
metadata_snapshot_manifest_payload = metadata_snapshot_payload["manifest"]

source_xml_snapshot_relative_path = metadata_snapshot_manifest_payload.get(
    "source_xml_snapshot_relative_path"
)
if not isinstance(source_xml_snapshot_relative_path, str) or not source_xml_snapshot_relative_path:
    raise RuntimeError(
        "Metadata snapshot manifest is missing source_xml_snapshot_relative_path."
    )

source_xml_file_name = metadata_snapshot_manifest_payload.get("source_xml_file_name")
if not isinstance(source_xml_file_name, str) or not source_xml_file_name:
    raise RuntimeError("Metadata snapshot manifest is missing source_xml_file_name.")

source_xml_snapshot_directory = (
    XML_SNAPSHOT_ROOT_DIRECTORY / source_xml_snapshot_relative_path
)
xml_file_path = source_xml_snapshot_directory / source_xml_file_name

metadata_manifest_payload = read_json_file(
    input_file_path=METADATA_MANIFEST_FILE_PATH,
)
qc_report_payload = read_json_file(input_file_path=QC_REPORT_FILE_PATH)
qc_report_file_sha256 = sha256_of_file(input_file_path=QC_REPORT_FILE_PATH)
qc_result = SimpleNamespace(**metadata_snapshot_manifest_payload["qc_summary"])

print(f"Resolved metadata snapshot directory: {metadata_snapshot_directory}")
print(f"Resolved metadata CSV path: {METADATA_CSV_FILE_PATH}")
print(f"Resolved metadata manifest path: {METADATA_MANIFEST_FILE_PATH}")
print(f"Resolved QC report path: {QC_REPORT_FILE_PATH}")
print(f"Resolved source XML file path: {xml_file_path}")

Resolved metadata snapshot directory: C:\pago-proj\pAgo-project\data\02-intermediate\protein_metadata_csv\latest
Resolved metadata CSV path: C:\pago-proj\pAgo-project\data\02-intermediate\protein_metadata_csv\latest\protein_metadata.csv
Resolved metadata manifest path: C:\pago-proj\pAgo-project\data\02-intermediate\protein_metadata_csv\latest\manifest.json
Resolved QC report path: C:\pago-proj\pAgo-project\data\02-intermediate\protein_metadata_csv\latest\qc_report.json
Resolved source XML file path: C:\pago-proj\pAgo-project\data\01-raw\protein_xml_snapshots\snapshots\2026-04-09T00-51-02Z__q_891f443d754c\protein_records.xml


In [5]:
# =============================================================================
# CELL 5 — Load frozen QC report
# =============================================================================

print("Frozen metadata QC report loaded successfully.")
print(f"QC report path: {QC_REPORT_FILE_PATH}")
print(f"QC report SHA-256: {qc_report_file_sha256}")
print(f"Row count audited: {qc_result.row_count}")
print(f"Column count audited: {qc_result.column_count}")

Frozen metadata QC report loaded successfully.
QC report path: C:\pago-proj\pAgo-project\data\02-intermediate\protein_metadata_csv\latest\qc_report.json
QC report SHA-256: 13f093ac59468a15f66166796e73b787597b9929b1a8894651ba7821adf8f172
Row count audited: 41345
Column count audited: 140


In [6]:
# =============================================================================
# CELL 6 — Print QC check summary
# =============================================================================

qc_checks = qc_report_payload["checks"]

print("QC check summary:")
for check_name, check_value in qc_checks.items():
    print(f"- {check_name}: {check_value}")

print(f"Empty protein_uid count: {qc_result.empty_protein_uid_count}")
print(f"Duplicate protein_uid count: {qc_result.duplicate_protein_uid_count}")
print(f"Fully empty column count: {qc_result.fully_empty_column_count}")
print(f"Normalization collision count: {qc_result.normalization_collision_count}")

QC check summary:
- protein_uid_has_no_duplicates: True
- protein_uid_has_no_empty_values: True
- row_count_matches_metadata_manifest: True
- row_count_matches_source_xml: True
- schema_matches_metadata_manifest: True
Empty protein_uid count: 0
Duplicate protein_uid count: 0
Fully empty column count: 0
Normalization collision count: 0


In [7]:
# =============================================================================
# CELL 7 — Inspect critical column completeness
# =============================================================================

critical_completeness_dataframe = pd.DataFrame.from_dict(
    qc_report_payload["critical_column_completeness"],
    orient="index",
).reset_index(names="column_name")

critical_completeness_dataframe.sort_values(
    by=["nonempty_fraction", "column_name"],
    ascending=[False, True],
)

,column_name,missing_count,nonempty_count,nonempty_fraction
0,feature__keys_present,0,41345,1.000000
1,gbseq__locus,0,41345,1.000000
2,protein_uid,0,41345,1.000000
4,taxonomy__raw,0,41345,1.000000
3,reference__count,692,40653,0.983263


In [8]:
# =============================================================================
# CELL 8 — Inspect flattening risks from source XML
# =============================================================================

flattening_risks = qc_report_payload.get("xml_flattening_risks", {})
normalization_collisions = flattening_risks.get("normalization_collisions", {})
repeated_feature_keys = flattening_risks.get("repeated_feature_keys", {})

print("Normalization collisions by category:")
print(json.dumps(normalization_collisions, indent=2, ensure_ascii=False))
print("Repeated feature keys in one GBSeq:")
print(json.dumps(repeated_feature_keys, indent=2, ensure_ascii=False))

Normalization collisions by category:
{}
Repeated feature keys in one GBSeq:
{
  "het": {
    "max_occurrences_in_one_row": 3,
    "rows_with_multiple_occurrences": 18
  },
  "non_std_res": {
    "max_occurrences_in_one_row": 19,
    "rows_with_multiple_occurrences": 2
  },
  "region": {
    "max_occurrences_in_one_row": 80,
    "rows_with_multiple_occurrences": 11556
  },
  "sec_str": {
    "max_occurrences_in_one_row": 60,
    "rows_with_multiple_occurrences": 229
  },
  "site": {
    "max_occurrences_in_one_row": 19,
    "rows_with_multiple_occurrences": 6610
  }
}


In [9]:
# =============================================================================
# CELL 9 — Inspect schema drift summary
# =============================================================================

schema_drift_summary = qc_report_payload.get("schema_drift")

if schema_drift_summary is None:
    print("No reference metadata manifest was configured, so no schema drift diff was generated.")
else:
    print(json.dumps(schema_drift_summary, indent=2, ensure_ascii=False))

No reference metadata manifest was configured, so no schema drift diff was generated.


In [10]:
# =============================================================================
# CELL 10 — Expose downstream variables
# =============================================================================

metadata_qc_output_directory = metadata_snapshot_directory
metadata_qc_report_file_path = QC_REPORT_FILE_PATH
metadata_qc_report_payload = qc_report_payload
metadata_qc_result = qc_result

print("Variables exposed for downstream notebooks:")
print("- metadata_qc_output_directory")
print("- metadata_qc_report_file_path")
print("- metadata_qc_report_payload")
print("- metadata_qc_result")

Variables exposed for downstream notebooks:
- metadata_qc_output_directory
- metadata_qc_report_file_path
- metadata_qc_report_payload
- metadata_qc_result
